# CoalGameRec local run notebook — Mac M4 Pro / 48GB RAM

This notebook runs an end-to-end **local executable prototype** of the CoalGameRec case-study pipeline:

1. load MovieLens-1M (automatic download);
2. optional Amazon Books 2018 loader if you provide `Books_5.json.gz`;
3. convert ratings to implicit positives;
4. build a temporal leave-one-out split;
5. train a small frozen BPR-MF backbone on Apple Silicon (`mps` if available);
6. cache full-catalogue base scores;
7. compute train-only item vectors;
8. compute post-hoc Shapley attributions on validation relevance;
9. apply fixed post-hoc reranking;
10. report HitRate@K and NDCG@K.

**Important:** this is a Mac-local implementation/prototype. It is not the validated official HCCF port required for confirmatory preregistration. The HCCF port, `PORT.md`, validation logs, lockfile/container, ethics determination, and external preregistration remain required real artifacts.

In [1]:
# Auto-reload local package while developing in VS Code/Jupyter.
%load_ext autoreload
%autoreload 2

# If needed, run once in your environment:
# %pip install -r ../requirements.txt

from pathlib import Path
import sys, json, platform

# Robust path setup whether the kernel cwd is code/ or code/notebooks/
CWD = Path.cwd().resolve()
CODE_DIR = CWD if (CWD / 'coalgamerec').exists() else CWD.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import numpy as np
import pandas as pd
import torch

from coalgamerec.data import load_movielens_1m, load_amazon_books_2018, preprocess_temporal_loo, item_user_vectors
from coalgamerec.models import TrainConfig, train_bprmf, cache_full_scores, pick_device
from coalgamerec.metrics import evaluate
from coalgamerec.attribution import compute_shapley_for_users
from coalgamerec.rerank import rerank_all
from coalgamerec.validation import assert_item_vector_isolation, assert_rerank_nonzero, assert_shapley_shapes
import coalgamerec, coalgamerec.rerank as _rerank_mod

print('coalgamerec package:', coalgamerec.__file__)
print('coalgamerec version:', getattr(coalgamerec, '__version__', 'unknown'))
print('rerank module:', _rerank_mod.__file__)
print('rerank sparse fix:', getattr(_rerank_mod, 'SPARSE_FIX_VERSION', 'MISSING - restart/pull required'))
print(platform.platform())
print('torch', torch.__version__)
print('mps available:', torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False)
print('device:', pick_device('auto'))

coalgamerec package: /Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/coalgamerec/__init__.py
coalgamerec version: 0.2.1-sparse-fix
rerank module: /Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/coalgamerec/rerank.py
rerank sparse fix: 0.2.1-sparse-safe
macOS-26.5.1-arm64-arm-64bit
torch 2.3.1
mps available: True
device: mps


## Configuration

Defaults are set for a quick Mac M4 Pro feasibility run. For a more complete MovieLens run, increase `SAMPLE_USERS`, `EPOCHS`, and `SHAPLEY_USERS`.

In [2]:
ROOT = CODE_DIR
DATA_RAW = ROOT / 'data' / 'raw'
RESULTS = ROOT / 'results' / 'mac_run'
RESULTS.mkdir(parents=True, exist_ok=True)

# Quick local defaults. Set SAMPLE_USERS=None for full MovieLens-1M.
SAMPLE_USERS = 2000
EPOCHS = 8
DIM = 64
BATCH_SIZE = 4096
SEED = 42

# Shapley is the expensive step. Use None for all users after feasibility testing.
SHAPLEY_USERS = 500
M_PERMUTATIONS = 64  # preregistration design says 128; 64 is faster for local smoke runs.
MAX_PLAYERS_PER_USER = 24  # set None for unbounded; 24 keeps Mac runs feasible
PLAYER_SELECTION = "similarity"
LAMBDA_ATTR = 0.10
KS = (5, 10, 20)

cfg = dict(SAMPLE_USERS=SAMPLE_USERS, EPOCHS=EPOCHS, DIM=DIM, BATCH_SIZE=BATCH_SIZE, SEED=SEED, SHAPLEY_USERS=SHAPLEY_USERS, M_PERMUTATIONS=M_PERMUTATIONS, MAX_PLAYERS_PER_USER=MAX_PLAYERS_PER_USER, PLAYER_SELECTION=PLAYER_SELECTION, LAMBDA_ATTR=LAMBDA_ATTR)
print(json.dumps(cfg, indent=2))

{
  "SAMPLE_USERS": 2000,
  "EPOCHS": 8,
  "DIM": 64,
  "BATCH_SIZE": 4096,
  "SEED": 42,
  "SHAPLEY_USERS": 500,
  "M_PERMUTATIONS": 64,
  "LAMBDA_ATTR": 0.1
}


## Load MovieLens-1M and build temporal leave-one-out split

In [3]:
ratings = load_movielens_1m(DATA_RAW)
print(ratings.head())
print('raw rows:', len(ratings), 'users:', ratings.user_raw.nunique(), 'items:', ratings.item_raw.nunique())

split, stats = preprocess_temporal_loo(ratings, name='ml1m', sample_users=SAMPLE_USERS, sample_seed=SEED)
print(json.dumps(stats, indent=2, default=str))
split.train.head()

   user_raw  item_raw  rating  timestamp  line_idx
0         1      1193       5  978300760         0
1         1       661       3  978302109         1
2         1       914       3  978301968         2
3         1      3408       4  978300275         3
4         1      2355       5  978824291         4
raw rows: 1000209 users: 6040 items: 3706
{
  "name": "ml1m",
  "users": 1991,
  "items": 2599,
  "train_interactions": 193789,
  "val_interactions": 1991,
  "test_interactions": 1991,
  "density_train": 0.037449979312446605,
  "mean_train_per_user": 97.33249623304872,
  "core_log": [
    {
      "iter": 0,
      "users": 1993,
      "items": 2599,
      "edges": 193884
    },
    {
      "iter": 1,
      "users": 1992,
      "items": 2599,
      "edges": 193880
    },
    {
      "iter": 2,
      "users": 1992,
      "items": 2599,
      "edges": 193880
    }
  ],
  "rating_threshold": 4.0,
  "min_uc": 5,
  "min_ic": 5
}


,user,item,timestamp,line_idx,user_raw,item_raw
130,354,141,978298124,130,2,1198
64,354,155,978298151,64,2,1210
88,354,240,978298261,88,2,1293
170,354,1459,978298372,170,2,2943
106,354,171,978298391,106,2,1225


## Optional: Amazon Books 2018 loader

Download `Books_5.json.gz` manually from the UCSD Amazon Reviews 2018 page, then set `AMAZON_BOOKS_5` below. The full file is very large; start with `max_rows` for a feasibility spike.

In [4]:
RUN_AMAZON = False
AMAZON_BOOKS_5 = DATA_RAW / 'Books_5.json.gz'

if RUN_AMAZON:
    amazon = load_amazon_books_2018(AMAZON_BOOKS_5, max_rows=2_000_000)
    amazon_split, amazon_stats = preprocess_temporal_loo(amazon, name='amazon_books_2018_sample', sample_users=50000, sample_seed=SEED)
    print(json.dumps(amazon_stats, indent=2, default=str))


## Train frozen backbone

The notebook uses a small BPR-MF backbone so the full pipeline runs on a Mac. Replace this with the validated HCCF port once that artifact exists.

In [5]:
train_cfg = TrainConfig(dim=DIM, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED, device='auto')
model = train_bprmf(split.train, split.n_users, split.n_items, train_cfg, verbose=True)
base_scores = cache_full_scores(model, split.n_users, batch_size=256)
print(base_scores.shape, base_scores.dtype)

train BPR-MF:   0%|          | 0/8 [00:00<?, ?it/s]

cache base scores:   0%|          | 0/8 [00:00<?, ?it/s]

(1991, 2599) float32


## Build train-only item vectors and evaluate base model

In [6]:
X_items = item_user_vectors(split.train_csr)
print('item vectors:', X_items.shape, 'nnz:', X_items.nnz, 'density:', X_items.nnz / (X_items.shape[0] * X_items.shape[1]))

base_summary, base_per_user = evaluate(base_scores, split, X_items, ks=KS)
print('item-vector isolation:', assert_item_vector_isolation(split))
pd.Series(base_summary, name='base').to_frame()

item vectors: (2599, 1991) nnz: 193789 density: 0.037449979312446605
item-vector isolation: {'item_vector_hash': 'db018f8b592b281af96336c9e056e49daa22c749c093de0f1935e11c9fb5238e', 'shape': (2599, 1991), 'nnz': 193789, 'density': 0.037449979312446605}


,base
HitRate@5,0.044701
NDCG@5,0.027703
HitRate@10,0.066801
NDCG@10,0.034815
HitRate@20,0.117529
NDCG@20,0.047594
Coverage@20,0.573682
ILD@20,0.706220


## Compute Shapley attributions using validation relevance

This is the expensive part. The notebook defaults to the first 500 users and `M=64` for speed. For a closer preregistration-like run, set `SHAPLEY_USERS=None` and `M_PERMUTATIONS=128`.

In [7]:
shapley = compute_shapley_for_users(
    split, base_scores, X_items,
    max_users=SHAPLEY_USERS,
    m=M_PERMUTATIONS,
    exact_threshold=8,
    seed=SEED,
    max_players_per_user=MAX_PLAYERS_PER_USER,
    player_selection=PLAYER_SELECTION,
    checkpoint_path=RESULTS / "shapley_notebook_checkpoint.npz",
    save_every=10,
    alpha=0.70, beta=0.30, lambda_pref=0.20, lambda_attr_value=0.10,
)
# Users without Shapley in quick mode get zero weights so the notebook can still evaluate all users.
print('computed users:', len(shapley))
print('shapley shapes:', assert_shapley_shapes(split, shapley))
first_u = next(iter(shapley))
print(first_u, shapley[first_u][:10])

Shapley users:   0%|          | 0/500 [00:00<?, ?it/s]

computed users: 500
shapley shapes: {'checked_users': 500, 'status': 'ok'}
0 [0.0796331  0.13395187 0.1358969  0.15499257 0.1532681  0.12950422
 0.1400622  0.1421482  0.13250057 0.11430774]


## Rerank and evaluate attribution families

In [8]:
print('rerank nonzero:', assert_rerank_nonzero(split, base_scores, X_items, family='uniform'))
families = ['uniform', 'additive-pref', 'attention', 'heuristic-pop', 'shapley-mc']
rows = []
for fam in families:
    scores = rerank_all(base_scores, split, X_items, fam, shapley_by_user=shapley, lambda_attr=LAMBDA_ATTR)
    summary, _ = evaluate(scores, split, X_items, ks=KS)
    summary['family'] = fam
    rows.append(summary)
results = pd.DataFrame(rows).set_index('family')
results.to_csv(RESULTS / 'ml1m_mac_local_results.csv')
results

rerank nonzero: {'inspected_users': 100, 'changed_users': 100, 'max_abs_delta': 2.7667551040649414}


,HitRate@5,NDCG@5,HitRate@10,NDCG@10,HitRate@20,NDCG@20,Coverage@20,ILD@20
family,,,,,,,,
uniform,0.043697,0.027720,0.071321,0.036717,0.116022,0.047855,0.524048,0.691809
additive-pref,0.044199,0.027814,0.071321,0.036630,0.115520,0.047663,0.523663,0.691372
attention,0.045706,0.028551,0.071321,0.036851,0.117027,0.048247,0.522124,0.690093
heuristic-pop,0.043697,0.027620,0.070819,0.036467,0.115520,0.047634,0.522124,0.691410
shapley-mc,0.044701,0.027847,0.068810,0.035585,0.118031,0.047918,0.561755,0.702338


## Reranking strength sensitivity

In [9]:
sens_rows = []
for lam in [0.05, 0.10, 0.20]:
    for fam in ['uniform', 'additive-pref', 'shapley-mc']:
        scores = rerank_all(base_scores, split, X_items, fam, shapley_by_user=shapley, lambda_attr=lam)
        summary, _ = evaluate(scores, split, X_items, ks=KS)
        summary.update({'family': fam, 'lambda_attr': lam})
        sens_rows.append(summary)
sensitivity = pd.DataFrame(sens_rows)
sensitivity.to_csv(RESULTS / 'ml1m_lambda_sensitivity.csv', index=False)
sensitivity

,HitRate@5,NDCG@5,HitRate@10,NDCG@10,HitRate@20,NDCG@20,Coverage@20,ILD@20,family,lambda_attr
0,0.044701,0.028087,0.068810,0.035896,0.115520,0.047630,0.545594,0.698730,uniform,0.05
1,0.044199,0.028002,0.068810,0.035990,0.115520,0.047713,0.545210,0.698467,additive-pref,0.05
2,0.044199,0.027653,0.067303,0.035116,0.117529,0.047747,0.566372,0.704153,shapley-mc,0.05
3,0.043697,0.027720,0.071321,0.036717,0.116022,0.047855,0.524048,0.691809,uniform,0.10
4,0.044199,0.027814,0.071321,0.036630,0.115520,0.047663,0.523663,0.691372,additive-pref,0.10
5,0.044701,0.027847,0.068810,0.035585,0.118031,0.047918,0.561755,0.702338,shapley-mc,0.10
6,0.047212,0.029806,0.074335,0.038407,0.118533,0.049407,0.484032,0.680424,uniform,0.20
7,0.047212,0.029904,0.073832,0.038397,0.119538,0.049772,0.480185,0.679545,additive-pref,0.20
8,0.044701,0.028111,0.069312,0.036020,0.119538,0.048587,0.557907,0.699228,shapley-mc,0.20


## Save run manifest

In [10]:
manifest = {
    'note': 'Mac-local CoalGameRec prototype; not confirmatory HCCF preregistration run',
    'config': cfg,
    'dataset_stats': stats,
    'torch': torch.__version__,
    'device': str(pick_device('auto')),
}
(RESULTS / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print('wrote', RESULTS)

wrote /Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/results/mac_run
